# 04 · Train XGBoost, Random Forest and SVM (MedSafe-AI · Component 3)

For every drug pair the models predict **which of the side effects** (about 60 labels) will be reported. Then we measure how good that is on data the model never saw.

**Fair-test rules used here**
1. Learn only from the **training** part.
2. Choose settings and the score cut-off on the **validation** part only.
3. Look at the **test** part for the final numbers only.
4. Always compare with two simple baselines: *most-common side effects* and *drug history* (no machine learning).
5. A **label-shuffle test** at the end checks that nothing leaks.

> Change `SPLIT_MODE` to `"unseen_drug"` and run again to get the harder result for new medicines.

In [ ]:
import sys, importlib.util, subprocess
from pathlib import Path

HERE = Path.cwd()                                   # the folder of this notebook (component3)
if not (HERE / "c3_utils.py").exists():
    if Path("/content").exists():                   # Google Colab: get the code from GitHub
        REPO_DIR = Path("/content/repo")
        if not (REPO_DIR / "component3" / "c3_utils.py").exists():
            subprocess.run(["rm", "-rf", str(REPO_DIR)])
            subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/sksivakajan/mac-doc.git", str(REPO_DIR)])
        HERE = REPO_DIR / "component3"
        import os; os.chdir(HERE)
        print("Colab: code taken from GitHub ->", HERE)
    else:                                           # Jupyter was started from another folder
        HERE = Path(r"D:\med\data set\component3")
sys.path.insert(0, str(HERE))

def need(*pairs):
    """Install the packages that are missing. pairs = (import name, pip name)."""
    missing = [pip for mod, pip in pairs if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", missing)
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
        except Exception as e:
            print("!! Could not install", missing, "->", e)

need(("pandas", "pandas"), ("numpy", "numpy"), ("matplotlib", "matplotlib"),
     ("sklearn", "scikit-learn"), ("xgboost", "xgboost"), ("joblib", "joblib"))

import os, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import joblib
import c3_utils as c3
warnings.filterwarnings("ignore")

WORK = HERE / "work"
RESULTS = HERE / "results"
RESULTS.mkdir(exist_ok=True)
SEED = c3.SEED

## Step 1 · Settings
| Setting | Meaning |
|---|---|
| `SPLIT_MODE` | `"pair"` (random pairs) or `"unseen_drug"` (drugs held out). |
| `XGB_DEVICE` | `"cpu"` or `"cuda"` (GPU, Colab only). |
| `TRAIN_CAP` | Use at most this many training pairs. Smaller = faster. Set `None` to use all (needs more RAM and hours of time). |
| `SVM_TRAIN_CAP` | The SVM is slow, so it uses fewer pairs. |
| `USE_DRUG_HISTORY` | Add "how often did each drug cause each side effect in training" as features. |
| `TUNE` | `True` tries 2 settings per model on a small sample and keeps the better one (slower). |

In [ ]:
SPLIT_MODE = "pair"            # "pair"  or  "unseen_drug"
TRAIN_CAP = 60_000
SVM_TRAIN_CAP = 30_000
USE_DRUG_HISTORY = True
TUNE = False
MODELS_TO_RUN = ["random_forest", "xgboost", "svm"]
SVM_KERNEL = "linear"          # "linear" (fast) or "rbf" (very slow; use SVM_TRAIN_CAP <= 10_000)
N_JOBS = os.cpu_count() or 1
XGB_DEVICE = "cpu"             # "cuda" = use a GPU (Google Colab with a GPU runtime makes XGBoost much faster)

PARAMS = {   # first entry is used when TUNE = False
    "random_forest": [dict(n_estimators=150, min_samples_leaf=5, max_features="sqrt"),
                      dict(n_estimators=150, min_samples_leaf=2, max_features="sqrt")],
    "xgboost":       [dict(n_estimators=150, max_depth=5, learning_rate=0.1),
                      dict(n_estimators=150, max_depth=7, learning_rate=0.05)],
    "svm":           [dict(C=0.05), dict(C=0.5)],
}
print("split:", SPLIT_MODE, "| cores:", N_JOBS)

## Step 2 · Load the data and the split

In [ ]:
pairs = pd.read_csv(WORK / "modeling_pairs.csv")
Y = np.load(WORK / "modeling_Y.npy")
effects = pd.read_csv(WORK / "modeling_effects.csv")
drugs = pd.read_csv(WORK / "drugs_final.csv", keep_default_na=False)
z = np.load(WORK / "drug_features.npz")
FP, CM = z["FP"], z["CM"]
sp = np.load(WORK / f"split_{SPLIT_MODE}.npz")
tr_all, va, te = sp["train"], sp["val"], sp["test"]
ia, ib = pairs["idx_a"].to_numpy(), pairs["idx_b"].to_numpy()

sev_rank = effects["severity"].map({"Low": 0, "Medium": 1, "High": 2}).fillna(-1).to_numpy().astype(int)
high_mask = sev_rank == 2
print(f"train {len(tr_all):,}   val {len(va):,}   test {len(te):,}   | side effects {Y.shape[1]} (High severity: {high_mask.sum()})")

rng = np.random.default_rng(SEED)
def cap(idx, n):
    return np.sort(rng.choice(idx, n, replace=False)) if (n and len(idx) > n) else idx
tr = cap(tr_all, TRAIN_CAP)
print(f"pairs used for training: {len(tr):,}")

## Step 3 · Build the features
For a pair (A, B) the input numbers are (the same for A+B and B+A):
* **Fingerprints:** `A OR B` and `A AND B` (2 × 1024 numbers).
* **Drug classes:** which classes either drug has, and which both have.
* **Drug history** (optional): for each side effect, how often drug A and drug B had it in **training pairs**. For a *training* row the pair's own answer is removed first (leave-one-out), so the model can not read its own label. Validation/test rows never take part in this.

In [ ]:
hist = c3.DrugHistory(n_drugs=len(drugs)).fit(ia[tr_all], ib[tr_all], Y[tr_all]) if USE_DRUG_HISTORY else None

def build_X(idx, is_train):
    a, b = ia[idx], ib[idx]
    parts = [c3.pair_fp_matrix(a, b, FP), CM[a] | CM[b], CM[a] & CM[b]]
    X = np.hstack(parts).astype(np.float32)
    if hist is not None:
        X = np.hstack([X, hist.transform(a, b, Y[idx] if is_train else None)])
    return X

t0 = time.time()
Xtr, Xva, Xte = build_X(tr, True), build_X(va, False), build_X(te, False)
Ytr, Yva, Yte = Y[tr], Y[va], Y[te]
print(f"features: {Xtr.shape[1]} per pair   | train {Xtr.shape}  val {Xva.shape}  test {Xte.shape}   ({time.time()-t0:.0f}s)")
print(f"memory: about {(Xtr.nbytes + Xva.nbytes + Xte.nbytes)/1e9:.1f} GB")

## Step 4 · Baselines (no machine learning)
* **prevalence** – every pair gets the same list: the most common side effects in training.
* **drug_history** – average of the two drugs' own history (needs no model, no fingerprints).

If a trained model can not beat these, it has not learned anything useful.

In [ ]:
results, val_scores, test_scores = [], {}, {}

def record(name, S_va, S_te, extra=None):
    thr = c3.best_threshold(Yva, S_va)                 # cut-off chosen on VALIDATION only
    rv = c3.evaluate(Yva, S_va, thr, high_mask)
    rt = c3.evaluate(Yte, S_te, thr, high_mask)
    row = {"model": name, "threshold": round(thr, 4), **{f"val_{k}": v for k, v in rv.items()},
           **{f"test_{k}": v for k, v in rt.items()}}
    if extra:
        row.update(extra)
    results.append(row)
    val_scores[name], test_scores[name] = S_va, S_te
    print(f"{name:16s} val micro-AUPRC {rv['micro_AUPRC']:.3f} | TEST micro-AUPRC {rt['micro_AUPRC']:.3f}  "
          f"P@3 {rt['P@3']:.3f}  P@10 {rt['P@10']:.3f}  micro-F1 {rt['micro_F1']:.3f}  High-recall {rt.get('recall_High_effects', float('nan')):.3f}")

base = c3.PrevalenceBaseline().fit(Xtr, Y[tr_all])
record("prevalence", base.predict_proba(Xva), base.predict_proba(Xte))

if hist is not None:
    K = Y.shape[1]
    def hist_score(idx):
        h = hist.transform(ia[idx], ib[idx], None)
        return (h[:, :K] + h[:, K:2 * K]) / 2
    record("drug_history", hist_score(va), hist_score(te))

## Step 5 · Train the three models
Time is printed for each one. Run this cell once; the models stay in memory for the next steps.

In [ ]:
def make_model(name, p):
    from sklearn.multiclass import OneVsRestClassifier
    if name == "random_forest":
        from sklearn.ensemble import RandomForestClassifier
        return RandomForestClassifier(n_jobs=N_JOBS, random_state=SEED, **p)
    if name == "xgboost":
        from xgboost import XGBClassifier
        return OneVsRestClassifier(XGBClassifier(tree_method="hist", device=XGB_DEVICE, subsample=0.8, colsample_bytree=0.5,
                                                 n_jobs=N_JOBS, eval_metric="logloss", random_state=SEED,
                                                 verbosity=0, **p))
    if name == "svm":
        if SVM_KERNEL == "linear":
            from sklearn.svm import LinearSVC
            base = LinearSVC(dual=False, random_state=SEED, **p)
        else:
            from sklearn.svm import SVC
            base = SVC(kernel="rbf", gamma="scale", **p)
        return OneVsRestClassifier(base, n_jobs=N_JOBS)
    raise ValueError(name)

trained = {}
for name in MODELS_TO_RUN:
    n_fit = min(len(Xtr), SVM_TRAIN_CAP) if name == "svm" else len(Xtr)
    Xf, Yf = Xtr[:n_fit], Ytr[:n_fit]
    params = PARAMS[name][0]
    t0 = time.time()
    if TUNE:                                   # pick the better setting on VALIDATION, using a small sample
        n_small = min(15_000, n_fit)
        best_p, best_s = None, -1
        for p in PARAMS[name]:
            m = make_model(name, p).fit(Xf[:n_small], Yf[:n_small])
            s = c3.evaluate(Yva, c3.predict_scores(m, Xva), 0.5)["micro_AUPRC"]
            print(f"   {name} {p} -> val micro-AUPRC {s:.3f}")
            if s > best_s:
                best_p, best_s = p, s
        params = best_p
    model = make_model(name, params).fit(Xf, Yf)
    secs = time.time() - t0
    trained[name] = model
    record(name, c3.predict_scores(model, Xva), c3.predict_scores(model, Xte),
           {"params": str(params), "train_pairs": n_fit, "train_seconds": round(secs)})
    print(f"   -> trained in {secs/60:.1f} min\n")

## Step 6 · Results table

In [ ]:
res = pd.DataFrame(results)
cols = ["model", "test_micro_AUPRC", "test_macro_AUPRC", "test_P@3", "test_P@5", "test_P@10", "test_R@10",
        "test_micro_F1", "test_macro_F1", "test_recall_High_effects", "val_micro_AUPRC"]
show = res[cols].round(3)
display(show)
res.to_csv(RESULTS / f"metrics_{SPLIT_MODE}.csv", index=False)

learned = res[~res["model"].isin(["prevalence", "drug_history"])]
best_name = learned.sort_values("val_micro_AUPRC", ascending=False)["model"].iloc[0]   # chosen on VALIDATION
print("Best trained model (by validation micro-AUPRC):", best_name)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, m, t in zip(ax, ["test_micro_AUPRC", "test_P@3", "test_micro_F1"], ["micro-AUPRC", "Precision@3", "micro-F1"]):
    a.bar(res["model"], res[m], color=["#999" if x in ("prevalence", "drug_history") else "#3b6ea5" for x in res["model"]])
    a.set_title(t + " (test)"); a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(RESULTS / f"comparison_{SPLIT_MODE}.png", dpi=130); plt.show()

## Step 7 · Sanity checks (do this before you trust the numbers)
1. **Overfitting gap:** training score far above test score means the model memorised.
2. **Label-shuffle test:** we shuffle the training answers and train again. A healthy pipeline must then fall back to the *prevalence* baseline. If it stays clearly **above** the baseline, labels are leaking into the features.

In [ ]:
# 1. overfitting gap on a sample of training rows
sample = np.arange(min(8000, len(Xtr)))
rows = []
for name, m in trained.items():
    s_tr = c3.evaluate(Ytr[sample], c3.predict_scores(m, Xtr[sample]), 0.5)["micro_AUPRC"]
    s_te = res.loc[res["model"] == name, "test_micro_AUPRC"].iloc[0]
    rows.append({"model": name, "train_micro_AUPRC": round(s_tr, 3), "test_micro_AUPRC": round(s_te, 3), "gap": round(s_tr - s_te, 3)})
display(pd.DataFrame(rows))
print("Note: a gap is normal for Random Forest (it sees its own training rows). Look at the TEST column for quality.")

# 2. label-shuffle test (fast Random Forest on a small sample)
from sklearn.ensemble import RandomForestClassifier
Ys = Y[tr_all].copy()
np.random.default_rng(1).shuffle(Ys, axis=0)                    # break the link between pairs and answers
hist_s = c3.DrugHistory(len(drugs)).fit(ia[tr_all], ib[tr_all], Ys) if USE_DRUG_HISTORY else None
sub = tr[:15_000]
pos = {int(p): i for i, p in enumerate(tr_all)}
Ys_sub = Ys[[pos[int(p)] for p in sub]]
def X_shuf(idx, own=None):
    a, b = ia[idx], ib[idx]
    X = np.hstack([c3.pair_fp_matrix(a, b, FP), CM[a] | CM[b], CM[a] & CM[b]]).astype(np.float32)
    return np.hstack([X, hist_s.transform(a, b, own)]) if hist_s is not None else X
rf = RandomForestClassifier(n_estimators=60, min_samples_leaf=5, n_jobs=N_JOBS, random_state=SEED).fit(X_shuf(sub, Ys_sub), Ys_sub)
shuf = c3.evaluate(Yte, c3.predict_scores(rf, X_shuf(te)), 0.5)["micro_AUPRC"]
basel = res.loc[res["model"] == "prevalence", "test_micro_AUPRC"].iloc[0]
print(f"label-shuffle model: test micro-AUPRC {shuf:.3f}   |   prevalence baseline {basel:.3f}")
print("PASS: no sign of leakage" if shuf < basel + 0.02 else "!! WARNING: shuffled model beats the baseline -> look for leakage")

## Step 8 · Severity (Low / Medium / High) for a pair
A pair gets the **worst severity among its side effects**. We compare the true severity (from the TWOSIDES signals) with the severity from the predicted side effects.

⚠ The severity of each side effect comes from the placeholder rules in `severity.py`. Agree a real source with your supervisor before using these numbers in the thesis.

⚠ About 28 of the side effects are marked High, so **most pairs contain at least one High effect**. "Worst severity" is therefore dominated by the High class. Talk with your supervisor about a better pair-level rule, for example *"High only if the predicted chance of a High effect is above a higher cut-off"*, or a severity **score** (the sum of the probabilities of High effects).

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score
thr = float(res.loc[res["model"] == best_name, "threshold"].iloc[0])
S_best = test_scores[best_name]
true_sev = c3.pair_severity(Yte, sev_rank)
pred_sev = c3.pair_severity(S_best >= thr, sev_rank)
ok = true_sev >= 0
labels = [0, 1, 2]
cm = confusion_matrix(true_sev[ok], np.where(pred_sev[ok] < 0, 0, pred_sev[ok]), labels=labels)
print(f"best model: {best_name} | pairs with a known severity: {ok.sum():,} of {len(ok):,}")
print("pair severity accuracy :", round(float((true_sev[ok] == np.where(pred_sev[ok] < 0, 0, pred_sev[ok])).mean()), 3))
print("pair severity macro-F1 :", round(float(f1_score(true_sev[ok], np.where(pred_sev[ok] < 0, 0, pred_sev[ok]), average="macro")), 3))
hi = true_sev == 2
print("High-severity pairs found (recall):", round(float((pred_sev[hi] == 2).mean()), 3) if hi.any() else "n/a")
display(pd.DataFrame(cm, index=["true Low", "true Medium", "true High"], columns=["pred Low", "pred Medium", "pred High"]))

## Step 9 · Try it: top predicted side effects for one pair

In [ ]:
def show_pair(name_a, name_b, model_name=best_name, top=8):
    names = drugs["drug_name"].str.lower().tolist()
    try:
        a, b = names.index(name_a.lower()), names.index(name_b.lower())
    except ValueError:
        print("drug not in the list; examples:", names[:10]); return
    X = np.hstack([c3.pair_fp_matrix(np.array([a]), np.array([b]), FP), CM[[a]] | CM[[b]], CM[[a]] & CM[[b]]]).astype(np.float32)
    if hist is not None:
        X = np.hstack([X, hist.transform(np.array([a]), np.array([b]), None)])
    s = c3.predict_scores(trained[model_name], X)[0]
    out = effects.assign(score=s).sort_values("score", ascending=False).head(top)
    display(out[["effect_name", "severity", "score"]].round(3))

show_pair("warfarin", "ibuprofen")

## Step 10 · Save the best model

In [ ]:
joblib.dump({"model": trained[best_name], "model_name": best_name, "threshold": thr, "effects": effects,
             "drugs": drugs[["drug_id", "drug_name", "drug_class", "main_use"]], "FP": FP, "CM": CM,
             "history": hist, "split_mode": SPLIT_MODE},
            RESULTS / f"best_model_{SPLIT_MODE}.joblib")
print("saved:", RESULTS / f"best_model_{SPLIT_MODE}.joblib")
other = "unseen_drug" if SPLIT_MODE == "pair" else "pair"
print(f"\nNow change SPLIT_MODE to '{other}' (Step 1) and run the notebook again to get the second result.")

## Step 11 · Both splits side by side (after you ran both)

In [ ]:
parts = []
for mode in ["pair", "unseen_drug"]:
    f = RESULTS / f"metrics_{mode}.csv"
    if f.exists():
        m = pd.read_csv(f)[["model", "test_micro_AUPRC", "test_P@3", "test_P@10", "test_micro_F1", "test_recall_High_effects"]]
        m.insert(0, "split", mode)
        parts.append(m)
if parts:
    both = pd.concat(parts, ignore_index=True).round(3)
    display(both)
    both.to_csv(RESULTS / "metrics_both_splits.csv", index=False)
else:
    print("No saved results yet.")

## How to read your results
* Compare each model with **prevalence** and **drug_history**. A real improvement is a higher `test_micro_AUPRC` and `P@3/P@10`.
* The `unseen_drug` result is lower; that is normal and honest. It shows how the system behaves for a **new** medicine.
* `recall_High_effects` tells how many serious side effects the model finds. For safety this matters more than overall F1.
* TWOSIDES is built from reports, so even a perfect model only learns *reported signals*, not proven causes.